# NUS ST3248 — Statistical Learning I
## Detailed case study with Palmer Penguins

This notebook is a systematic, student-oriented walkthrough of the core ideas in **ST3248 Statistical Learning I** using one coherent online dataset.

We will cover:

1. statistical-learning foundations,
2. exploratory data analysis,
3. simple and multiple linear regression,
4. residual diagnostics,
5. bias–variance trade-off,
6. validation and K-fold cross-validation,
7. bootstrap,
8. best subset selection,
9. ridge and lasso regularisation,
10. logistic regression,
11. LDA and QDA,
12. K-nearest neighbours,
13. multiclass evaluation,
14. PCA,
15. K-means clustering,
16. hierarchical clustering.

All visualisations and result tables use **Bokeh**.

The common statistical-learning framework is

$$
Y=f(X)+\\epsilon
$$

and our aim is to estimate

$$
\\hat f(X) \\approx f(X).
$$

The deeper question throughout is:

> How much model complexity can the available data support before generalisation begins to deteriorate?

## Why Palmer Penguins?

The Palmer Penguins dataset contains measurements of Adelie, Chinstrap and Gentoo penguins.

It lets us study multiple ST3248 tasks using the same observations.

### Regression

$$
Y=\\text{body mass}
$$

### Classification

$$
Y=\\text{species}
$$

### Unsupervised learning

$$
X=\\{\\text{bill dimensions, flipper length, body mass}\\}.
$$

This makes the transitions between methods much easier to understand.

# 1. Setup

In [1]:
# %pip install -q pandas numpy scipy scikit-learn bokeh

In [2]:
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, label_binarize
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, roc_curve, auc, silhouette_score
)

from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, DataTable, TableColumn, HoverTool, Div, LinearColorMapper, ColorBar
from bokeh.palettes import Category10, Viridis256, RdBu11
from bokeh.plotting import figure
from bokeh.transform import transform

warnings.filterwarnings('ignore')
RANDOM_STATE=42
output_notebook()
print('Environment ready.')

Loading BokehJS ...

Environment ready.


In [3]:
from bokeh.models import Div
from bokeh.io import show
import numpy as np


def show_table(
    df,
    title=None,
    width=950,
    height=None,
    round_digits=3
):
    """
    Reliable Bokeh table renderer using Div + HTML.

    Works well in:
    - Jupyter Notebook
    - JupyterLab
    - VS Code notebooks
    - Polyglot Jupyter environments

    Parameters
    ----------
    df : pd.DataFrame
        Table to render.

    title : str, optional
        Table title.

    width : int
        Bokeh Div width.

    height : ignored
        Kept for compatibility with existing notebook calls.

    round_digits : int
        Number of decimal places for numeric columns.
    """

    x = df.copy()

    # -------------------------------------------------
    # 1. Make column names safe
    # -------------------------------------------------
    x.columns = [str(c) for c in x.columns]

    # -------------------------------------------------
    # 2. Round numeric values
    # -------------------------------------------------
    numeric_cols = x.select_dtypes(
        include=np.number
    ).columns

    x[numeric_cols] = x[numeric_cols].round(
        round_digits
    )

    # -------------------------------------------------
    # 3. Convert dataframe → HTML
    # -------------------------------------------------
    table_html = x.to_html(
        index=False,
        border=0,
        classes="st3248-table"
    )

    title_html = (
        f"<h3>{title}</h3>"
        if title
        else ""
    )

    # -------------------------------------------------
    # 4. Styling
    # -------------------------------------------------
    html = f"""
    <style>

    .st3248-wrapper {{
        font-family:
            -apple-system,
            BlinkMacSystemFont,
            "Segoe UI",
            Arial,
            sans-serif;

        margin: 5px 0 20px 0;
    }}

    .st3248-wrapper h3 {{
        margin-bottom: 12px;
    }}

    .st3248-table {{
        border-collapse: collapse;
        width: 100%;
        font-size: 14px;
        background: white;
    }}

    .st3248-table thead th {{
        background: #f1f5f9;
        padding: 10px 14px;
        text-align: left;
        font-weight: 600;

        border-top: 1px solid #cbd5e1;
        border-bottom: 2px solid #94a3b8;
    }}

    .st3248-table tbody td {{
        padding: 10px 14px;
        border-bottom: 1px solid #e2e8f0;
    }}

    .st3248-table tbody tr:hover {{
        background: #f8fafc;
    }}

    .st3248-table tbody tr:nth-child(even) {{
        background: #fafafa;
    }}

    </style>

    <div class="st3248-wrapper">

        {title_html}

        {table_html}

    </div>
    """

    show(
        Div(
            text=html,
            width=width
        )
    )


def show_note(title, body, width=900):
    html = (
        "<div style='border-left:5px solid #3b82f6;padding:12px 16px;"
        "background:#f8fafc;border-radius:6px;font-family:Arial;'>"
        f"<h3 style='margin:0 0 6px 0'>{title}</h3>"
        f"<div style='line-height:1.5'>{body}</div></div>"
    )
    show(Div(text=html, width=width))


def regression_metrics(y_true, y_pred):
    return {
        'RMSE': mean_squared_error(y_true, y_pred)**0.5,
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred)
    }


def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Macro_F1': f1_score(y_true, y_pred, average='macro')
    }

# 2. Load the online dataset

One row represents one observed penguin.

Before modelling, inspect:

- shape,
- columns and data types,
- missing values,
- possible response variables,
- suspicious shortcut predictors.

In [4]:
DATA_URLS=[
    'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv',
    'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
]

df=None
for url in DATA_URLS:
    try:
        df=pd.read_csv(url, na_values=['NA',''])
        print('Loaded:', url)
        break
    except Exception:
        pass

if df is None:
    raise RuntimeError('Dataset download failed. Check internet access and rerun.')

print('Shape:', df.shape)
display(df.head())

Loaded: https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv
Shape: (344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [5]:
data_dictionary=pd.DataFrame({
    'variable':['species','island','bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g','sex','year'],
    'type':['categorical','categorical','continuous','continuous','continuous','continuous','categorical','integer'],
    'meaning':['Penguin species','Observation island','Bill length','Bill depth','Flipper length','Body mass','Sex','Observation year']
})
show_table(data_dictionary, 'Data dictionary', height=250)

In [6]:
quality=pd.DataFrame({
    'column':df.columns,
    'dtype':df.dtypes.astype(str).values,
    'missing_n':df.isna().sum().values,
    'missing_pct':100*df.isna().mean().values,
    'n_unique':df.nunique(dropna=True).values
})
show_table(quality, 'Data-quality audit', height=300)

## Leakage principle

A transformation learned from data must be fitted using training observations only.

Incorrect:

$$
\\text{scale all data} \\rightarrow \\text{split}.
$$

Correct:

$$
\\text{split} \\rightarrow \\text{fit scaler on training data} \\rightarrow \\text{transform validation/test data}.
$$

The same applies to imputation, PCA and feature selection.

# 3. Exploratory data analysis

In [7]:
species_counts=df['species'].value_counts().rename_axis('species').reset_index(name='count')
p=figure(x_range=species_counts['species'].tolist(), width=720, height=380, title='Species counts', toolbar_location=None)
p.vbar(x=species_counts['species'], top=species_counts['count'], width=0.7)
p.xaxis.axis_label='Species'; p.yaxis.axis_label='Count'
p.add_tools(HoverTool(tooltips=[('species','@x'),('count','@top')]))
show(p)
show_note('Interpretation','Class frequencies are not identical, so later we compare accuracy with balanced accuracy and macro-F1.')

In [8]:
plot_df=df.dropna(subset=['flipper_length_mm','body_mass_g','species']).copy()
species_order=sorted(plot_df['species'].unique())
palette=Category10[10]
species_color={s:palette[i] for i,s in enumerate(species_order)}

p=figure(width=880,height=500,title='Body mass vs flipper length',x_axis_label='Flipper length (mm)',y_axis_label='Body mass (g)')
for s in species_order:
    part=plot_df[plot_df['species']==s]
    p.scatter('flipper_length_mm','body_mass_g',source=ColumnDataSource(part),size=8,alpha=0.65,color=species_color[s],legend_label=s)
p.legend.location='top_left'
p.add_tools(HoverTool(tooltips=[('species','@species'),('flipper','@flipper_length_mm'),('mass','@body_mass_g')]))
show(p)

The pooled relationship is strongly positive:

$$
\\text{longer flipper} \\Rightarrow \\text{larger body mass on average}.
$$

But species form distinct groups. This warns us that a global prediction relationship may hide subgroup structure.

In [9]:
num_cols=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
corr=df[num_cols].corr()
corr_long=corr.stack().rename('correlation').reset_index().rename(columns={'level_0':'x','level_1':'y'})
mapper=LinearColorMapper(palette=RdBu11[::-1],low=-1,high=1)
p=figure(x_range=num_cols,y_range=list(reversed(num_cols)),width=760,height=540,title='Correlation matrix',toolbar_location=None,tools='hover',tooltips=[('pair','@x × @y'),('correlation','@correlation{0.000}')])
p.rect(x='x',y='y',width=1,height=1,source=ColumnDataSource(corr_long),fill_color=transform('correlation',mapper),line_color='white')
p.xaxis.major_label_orientation=0.8
p.add_layout(ColorBar(color_mapper=mapper),'right')
show(p)

# 4. Simple linear regression

We first predict body mass from flipper length:

$$
Y_i=\\beta_0+\\beta_1X_i+\\epsilon_i.
$$

OLS chooses coefficients by minimising

$$
RSS=\\sum_{i=1}^{n}(y_i-\\hat y_i)^2.
$$

In [10]:
reg_df=df[['flipper_length_mm','body_mass_g']].dropna()
X=reg_df[['flipper_length_mm']]; y=reg_df['body_mass_g']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=RANDOM_STATE)
lr=LinearRegression().fit(X_train,y_train)
train_pred=lr.predict(X_train); test_pred=lr.predict(X_test)
show_table(pd.DataFrame([
    {'split':'train',**regression_metrics(y_train,train_pred)},
    {'split':'test',**regression_metrics(y_test,test_pred)}
]),'Simple linear regression performance',height=170)
print('Intercept:',round(lr.intercept_,3))
print('Slope:',round(lr.coef_[0],3),'grams per mm')

Intercept: -5741.265
Slope: 49.506 grams per mm


The slope is an **association**, not a causal effect.

The test RMSE is more relevant to generalisation than the training RMSE.

In [11]:
grid=pd.DataFrame({'flipper_length_mm':np.linspace(reg_df['flipper_length_mm'].min(),reg_df['flipper_length_mm'].max(),200)})
grid_pred=lr.predict(grid)
test_points=X_test.copy(); test_points['body_mass_g']=y_test.values
p=figure(width=860,height=480,title='Simple regression fit',x_axis_label='Flipper length (mm)',y_axis_label='Body mass (g)')
p.scatter('flipper_length_mm','body_mass_g',source=ColumnDataSource(test_points),size=8,alpha=0.7,legend_label='test observations')
p.line(grid['flipper_length_mm'],grid_pred,line_width=3,legend_label='fitted line')
p.legend.location='top_left'; show(p)

# 5. Residual diagnostics

Residual:

$$
e_i=y_i-\\hat y_i.
$$

A patternless cloud around zero is desirable. Curvature can indicate nonlinearity; a funnel shape can indicate non-constant variance.

In [38]:
resid_df=pd.DataFrame({'fitted':test_pred,'residual':y_test.values-test_pred})
p=figure(width=860,height=420,title='Residuals vs fitted values',x_axis_label='Fitted body mass',y_axis_label='Residual')
p.scatter('fitted','residual',source=ColumnDataSource(resid_df),size=8,alpha=0.7)
p.line([resid_df['fitted'].min(),resid_df['fitted'].max()],[0,0],line_dash='dashed',line_width=2)
show(p)

# 6. Multiple regression and preprocessing pipelines

Now use several predictors:

$$
Y=\\beta_0+\\beta_1X_1+\\cdots+\\beta_pX_p+\\epsilon.
$$

Numerical missing values are imputed with medians; categorical missing values use the mode; categoricals are one-hot encoded.

Everything is fitted inside a pipeline to prevent leakage.

In [13]:
num_features=['bill_length_mm','bill_depth_mm','flipper_length_mm','year']
cat_features=['sex','island']
work=df[num_features+cat_features+['body_mass_g']].copy()
work=work[work['body_mass_g'].notna()]
X=work[num_features+cat_features]; y=work['body_mass_g']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=RANDOM_STATE)
preprocess=ColumnTransformer([
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median'))]),num_features),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(drop='first',handle_unknown='ignore'))]),cat_features)
])
multi_lr=Pipeline([('prep',preprocess),('model',LinearRegression())]).fit(X_train,y_train)
pred=multi_lr.predict(X_test)
show_table(pd.DataFrame([{'model':'multiple regression',**regression_metrics(y_test,pred)}]),'Multiple regression performance',height=150)

# 7. Bias–variance trade-off with polynomial regression

Polynomial regression increases flexibility:

$$
Y=\\beta_0+\\beta_1X+\\beta_2X^2+\\cdots+\\beta_dX^d+\\epsilon.
$$

Expected test error can be thought of as

$$
\\text{bias}^2+\\text{variance}+\\text{irreducible noise}.
$$

We therefore choose degree using cross-validation rather than training error.

In [14]:
poly_df=df[['flipper_length_mm','body_mass_g']].dropna()
X_poly=poly_df[['flipper_length_mm']]; y_poly=poly_df['body_mass_g']
cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
rows=[]
for degree in range(1,11):
    model=Pipeline([('poly',PolynomialFeatures(degree=degree,include_bias=False)),('scale',StandardScaler()),('model',LinearRegression())])
    rmse=np.sqrt(-cross_val_score(model,X_poly,y_poly,cv=cv,scoring='neg_mean_squared_error'))
    rows.append({'degree':degree,'CV_RMSE_mean':rmse.mean(),'CV_RMSE_sd':rmse.std(ddof=1)})
poly_results=pd.DataFrame(rows)
show_table(poly_results,'Polynomial degree vs CV error',height=320)

In [15]:
p=figure(width=820,height=420,title='Polynomial degree vs 10-fold CV RMSE',x_axis_label='Polynomial degree',y_axis_label='CV RMSE')
p.line(poly_results['degree'],poly_results['CV_RMSE_mean'],line_width=3)
p.scatter(poly_results['degree'],poly_results['CV_RMSE_mean'],size=9)
show(p)

# 8. K-fold cross-validation

$$
CV_K=\\frac{1}{K}\\sum_{k=1}^{K} Error_k.
$$

Cross-validation reduces dependence on one arbitrary train/validation split and is the workhorse for model selection.

In [16]:
cv_summary=[]
for k in [3,5,10]:
    cv_k=KFold(n_splits=k,shuffle=True,random_state=RANDOM_STATE)
    rmse=np.sqrt(-cross_val_score(LinearRegression(),X_poly,y_poly,cv=cv_k,scoring='neg_mean_squared_error'))
    cv_summary.append({'K':k,'mean_RMSE':rmse.mean(),'sd_RMSE':rmse.std(ddof=1)})
show_table(pd.DataFrame(cv_summary),'K-fold CV comparison',height=180)

# 9. Bootstrap

Bootstrap approximates sampling uncertainty by repeatedly sampling observations **with replacement** and recomputing an estimator.

Here we bootstrap the regression slope.

In [32]:
rng=np.random.default_rng(RANDOM_STATE)
boot_df=df[['flipper_length_mm','body_mass_g']].dropna().reset_index(drop=True)
slopes=[]
for _ in range(1000):
    idx=rng.integers(0,len(boot_df),len(boot_df))
    sample=boot_df.iloc[idx]
    slopes.append(LinearRegression().fit(sample[['flipper_length_mm']],sample['body_mass_g']).coef_[0])
slopes=np.asarray(slopes)
ci_low,ci_high=np.percentile(slopes,[2.5,97.5])
hist,edges=np.histogram(slopes,bins=35)
hist_df=pd.DataFrame({'left':edges[:-1],'right':edges[1:],'count':hist})
p=figure(width=850,height=420,title='Bootstrap distribution of slope',x_axis_label='Slope',y_axis_label='Frequency')
p.quad(top='count',bottom=0,left='left',right='right',source=ColumnDataSource(hist_df),alpha=0.65)
p.line([ci_low,ci_low],[0,hist.max()],line_dash='dashed',line_width=2)
p.line([ci_high,ci_high],[0,hist.max()],line_dash='dashed',line_width=2)
show(p)
print('95% bootstrap percentile interval:',(round(ci_low,3),round(ci_high,3)))

95% bootstrap percentile interval: (46.897, 52.523)


# 10. Best subset selection

For $p$ predictors, exhaustive subset selection considers

$$
2^p-1
$$

non-empty subsets, so its computational growth is

$$
\\Theta(2^p).
$$

We keep the candidate set intentionally small and choose subsets using CV RMSE.

In [18]:
sel_df=df[['body_mass_g','bill_length_mm','bill_depth_mm','flipper_length_mm','year','sex']].dropna().copy()
X_sel=pd.get_dummies(sel_df.drop(columns='body_mass_g'),drop_first=True,dtype=float)
y_sel=sel_df['body_mass_g']
cv=KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
subset_rows=[]
for r in range(1,X_sel.shape[1]+1):
    for cols in combinations(X_sel.columns,r):
        rmse=np.sqrt(-cross_val_score(LinearRegression(),X_sel[list(cols)],y_sel,cv=cv,scoring='neg_mean_squared_error')).mean()
        subset_rows.append({'n_features':r,'features':', '.join(cols),'CV_RMSE':rmse})
subset_results=pd.DataFrame(subset_rows)
best_by_size=subset_results.sort_values('CV_RMSE').groupby('n_features',as_index=False).first()
show_table(best_by_size,'Best subset at each model size',height=300)

# 11. Ridge and Lasso regularisation

Ridge:

$$
\\min_\\beta \\left[RSS+\\lambda\\sum_j\\beta_j^2\\right].
$$

Lasso:

$$
\\min_\\beta \\left[RSS+\\lambda\\sum_j|\\beta_j|\\right].
$$

Ridge shrinks coefficients. Lasso can additionally set coefficients exactly to zero.

In [48]:
alphas=np.logspace(-3,4,40)
cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
ridge_rows=[]; lasso_rows=[]
for alpha in alphas:
    ridge=Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=alpha))])
    lasso=Pipeline([('scale',StandardScaler()),('model',Lasso(alpha=alpha,max_iter=20000))])
    ridge_rmse=np.sqrt(-cross_val_score(ridge,X_sel,y_sel,cv=cv,scoring='neg_mean_squared_error')).mean()
    lasso_rmse=np.sqrt(-cross_val_score(lasso,X_sel,y_sel,cv=cv,scoring='neg_mean_squared_error')).mean()
    ridge_rows.append({'alpha':alpha,'CV_RMSE':ridge_rmse})
    lasso_rows.append({'alpha':alpha,'CV_RMSE':lasso_rmse})
ridge_df=pd.DataFrame(ridge_rows); lasso_df=pd.DataFrame(lasso_rows)
p=figure(x_axis_type='log',width=850,height=430,title='Regularisation strength vs CV RMSE',x_axis_label='alpha / lambda',y_axis_label='CV RMSE')
p.line(ridge_df['alpha'],ridge_df['CV_RMSE'],line_width=3,legend_label='Ridge',line_color=(20, 120, 10, 0.4))
p.line(lasso_df['alpha'],lasso_df['CV_RMSE'],line_width=3,legend_label='Lasso',line_color=(255, 0, 0, 0.4))
p.legend.location='top_left'; show(p)

# 12. Classification — predicting species

We now set

$$
Y\\in\\{\\text{Adelie},\\text{Chinstrap},\\text{Gentoo}\\}
$$

using four morphological measurements.

We compare logistic regression, LDA, QDA and KNN.

In [20]:
class_features=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
class_df=df[class_features+['species']].dropna().copy()
Xc=class_df[class_features]; yc=class_df['species']
Xc_train,Xc_test,yc_train,yc_test=train_test_split(Xc,yc,test_size=0.25,stratify=yc,random_state=RANDOM_STATE)
models={
    'Logistic Regression':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000))]),
    'LDA':Pipeline([('scale',StandardScaler()),('model',LinearDiscriminantAnalysis())]),
    'QDA':Pipeline([('scale',StandardScaler()),('model',QuadraticDiscriminantAnalysis(reg_param=0.01))]),
    'KNN (K=7)':Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(n_neighbors=7))])
}
skf=StratifiedKFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
rows=[]
for name,model in models.items():
    acc=cross_val_score(model,Xc,yc,cv=skf,scoring='accuracy')
    bal=cross_val_score(model,Xc,yc,cv=skf,scoring='balanced_accuracy')
    f1=cross_val_score(model,Xc,yc,cv=skf,scoring='f1_macro')
    rows.append({'model':name,'accuracy_mean':acc.mean(),'balanced_accuracy_mean':bal.mean(),'macro_F1_mean':f1.mean()})
benchmark=pd.DataFrame(rows).sort_values('macro_F1_mean',ascending=False).reset_index(drop=True)
show_table(benchmark,'Classifier comparison',height=230)

### LDA

$$
X\\mid Y=k \\sim N(\\mu_k,\\Sigma).
$$

### QDA

$$
X\\mid Y=k \\sim N(\\mu_k,\\Sigma_k).
$$

QDA is more flexible but usually has higher variance.

### KNN

$$
P(Y=j\\mid X=x_0)\\approx\\frac{1}{K}\\sum_{i\\in N_0}I(y_i=j).
$$

Small $K$ means low bias/high variance; large $K$ means higher bias/lower variance.

In [49]:
best_name=benchmark.loc[0,'model']
best_model=clone(models[best_name]).fit(Xc_train,yc_train)
pred=best_model.predict(Xc_test)
show_table(pd.DataFrame([{'model':best_name,**classification_metrics(yc_test,pred)}]),'Hold-out classification performance',height=150)
labels=sorted(yc.unique())
cm=confusion_matrix(yc_test,pred,labels=labels)
cm_long=pd.DataFrame([{'actual':a,'predicted':p_,'count':int(cm[i,j])} for i,a in enumerate(labels) for j,p_ in enumerate(labels)])
mapper=LinearColorMapper(palette=Viridis256,low=0,high=max(1,int(cm.max())))
p=figure(x_range=labels,y_range=list(reversed(labels)),width=640,height=500,title=f'Confusion matrix — {best_name}',x_axis_label='Predicted',y_axis_label='Actual',tools='hover',tooltips=[('actual','@actual'),('predicted','@predicted'),('count','@count')])
p.rect(x='predicted',y='actual',width=1,height=1,source=ColumnDataSource(cm_long),fill_color=transform('count',mapper),line_color='white')
p.add_layout(ColorBar(color_mapper=mapper),'right'); show(p)

# 13. Tune KNN

Because KNN uses Euclidean distances, standardisation is essential.

$$
d(x,z)=\\sqrt{\\sum_j(x_j-z_j)^2}.
$$

We select $K$ using CV macro-F1.

In [43]:
knn_rows=[]
for k in range(1,42,2):
    model=Pipeline([('scale',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=k))])
    score=cross_val_score(model,Xc,yc,cv=skf,scoring='f1_macro')
    knn_rows.append({'K':k,'macro_F1_mean':score.mean(),'macro_F1_sd':score.std(ddof=1)})
knn_df=pd.DataFrame(knn_rows)
p=figure(width=840,height=420,title='KNN neighbourhood size vs CV macro-F1',x_axis_label='K',y_axis_label='Macro-F1')
p.line(knn_df['K'],knn_df['macro_F1_mean'],line_width=3)
p.scatter(knn_df['K'],knn_df['macro_F1_mean'],size=8)
show(p)
print('Best K:',int(knn_df.loc[knn_df['macro_F1_mean'].idxmax(),'K']))

Best K: 3


# 14. Multiclass ROC curves

For each class, form a one-vs-rest classification problem.

$$
TPR=\\frac{TP}{TP+FN}
$$

and

$$
FPR=\\frac{FP}{FP+TN}.
$$

In [45]:
roc_model=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000))]).fit(Xc_train,yc_train)
proba=roc_model.predict_proba(Xc_test)
classes=roc_model.named_steps['model'].classes_
y_bin=label_binarize(yc_test,classes=classes)
p=figure(width=760,height=500,title='One-vs-rest ROC curves',x_axis_label='False positive rate',y_axis_label='True positive rate')
for i,cls in enumerate(classes):
    fpr,tpr,_=roc_curve(y_bin[:,i],proba[:,i])
    score_auc=auc(fpr,tpr)
    p.line(fpr,tpr,line_width=3,legend_label=f'{cls} AUC={score_auc:.3f}')
p.line([0,1],[0,1],line_dash='dashed',line_width=2)
p.legend.location='bottom_right'; show(p)

# 15. Principal Component Analysis

PCA creates orthogonal linear combinations of standardized predictors.

$$
Z_1=\\phi_{11}X_1+\\cdots+\\phi_{p1}X_p.
$$

The first PC explains the largest possible variance; later PCs explain remaining orthogonal variance.

In [24]:
pca_df=df[class_features+['species']].dropna().copy()
scaler=StandardScaler(); X_scaled=scaler.fit_transform(pca_df[class_features])
pca=PCA(); scores=pca.fit_transform(X_scaled)
pve=pca.explained_variance_ratio_
pve_df=pd.DataFrame({'component':[f'PC{i+1}' for i in range(len(pve))],'explained_variance_ratio':pve,'cumulative_explained_variance':np.cumsum(pve)})
show_table(pve_df,'PCA explained variance',height=220)

In [25]:
loadings=pd.DataFrame(pca.components_.T,index=class_features,columns=[f'PC{i+1}' for i in range(len(class_features))]).reset_index(names='feature')
show_table(loadings,'PCA loadings',height=220)

In [39]:
pca_plot=pd.DataFrame({'PC1':scores[:,0],'PC2':scores[:,1],'species':pca_df['species'].values})
p=figure(width=860,height=500,title='PCA projection',x_axis_label=f'PC1 ({100*pve[0]:.1f}% variance)',y_axis_label=f'PC2 ({100*pve[1]:.1f}% variance)')
for s in sorted(pca_plot['species'].unique()):
    part=pca_plot[pca_plot['species']==s]
    p.scatter('PC1','PC2',source=ColumnDataSource(part),size=8,alpha=0.7,color=species_color[s],legend_label=s)
p.legend.location='top_left'; show(p)

Species labels were **not** used to fit PCA. They are added afterward only for interpretation.

# 16. K-means clustering

K-means minimises within-cluster squared distance:

$$
\\sum_{k=1}^{K}\\sum_{i\\in C_k}\\|x_i-\\mu_k\\|^2.
$$

A useful complexity intuition for Lloyd's algorithm is

$$
O(nKpI).
$$

In [27]:
cluster_rows=[]
for k in range(2,8):
    km=KMeans(n_clusters=k,n_init=30,random_state=RANDOM_STATE)
    labels_k=km.fit_predict(X_scaled)
    cluster_rows.append({'K':k,'inertia':km.inertia_,'silhouette':silhouette_score(X_scaled,labels_k)})
cluster_eval=pd.DataFrame(cluster_rows)
show_table(cluster_eval,'K-means diagnostics',height=230)

In [46]:
p1=figure(width=780,height=380,title='K-means inertia',x_axis_label='K',y_axis_label='Inertia')
p1.line(cluster_eval['K'],cluster_eval['inertia'],line_width=3); p1.scatter(cluster_eval['K'],cluster_eval['inertia'],size=8)
p2=figure(width=780,height=380,title='Silhouette score',x_axis_label='K',y_axis_label='Silhouette')
p2.line(cluster_eval['K'],cluster_eval['silhouette'],line_width=3); p2.scatter(cluster_eval['K'],cluster_eval['silhouette'],size=8)
show(column(p1,p2))

In [29]:
km3=KMeans(n_clusters=3,n_init=30,random_state=RANDOM_STATE)
cluster3=km3.fit_predict(X_scaled)
cluster_table=pd.crosstab(pca_df['species'],cluster3,rownames=['species'],colnames=['cluster']).reset_index()
show_table(cluster_table,'Clusters vs species — post-hoc interpretation only',height=190)



In [30]:
display(cluster_table)

print(cluster_table.columns)
print(cluster_table.shape)

cluster,species,0,1,2
0,Adelie,24,0,127
1,Chinstrap,63,0,5
2,Gentoo,0,123,0


Index(['species', 0, 1, 2], dtype='object', name='cluster')
(3, 4)


# 17. Hierarchical clustering

Agglomerative clustering starts with one cluster per observation and repeatedly merges the nearest clusters.

We use Ward linkage, which prefers merges that minimally increase within-cluster variation.

In [47]:
rng=np.random.default_rng(RANDOM_STATE)
sample_idx=rng.choice(len(X_scaled),size=55,replace=False)
Z=linkage(X_scaled[sample_idx],method='ward')
d=dendrogram(Z,no_plot=True)
p=figure(width=950,height=460,title='Ward hierarchical-clustering dendrogram',x_axis_label='Sample order',y_axis_label='Merge distance')
for xs,ys in zip(d['icoord'],d['dcoord']):
    p.line(xs,ys,line_width=2)
p.xaxis.visible=False; show(p)

# 18. Final synthesis

| Method | Main purpose | Main trade-off |
|---|---|---|
| Linear regression | interpretable continuous prediction | may underfit |
| Polynomial regression | nonlinear flexibility | variance can rise |
| Cross-validation | estimate generalisation | more computation |
| Bootstrap | estimate sampling uncertainty | assumes resampling unit is appropriate |
| Best subset | select predictors | exponential search |
| Ridge | coefficient shrinkage | introduces bias |
| Lasso | shrinkage + sparsity | may remove correlated useful predictors |
| Logistic regression | probabilistic classification | linear log-odds structure |
| LDA | lower-variance generative classifier | shared covariance assumption |
| QDA | more flexible classifier | higher variance |
| KNN | local non-parametric classification | distance/scale sensitivity |
| PCA | dimension reduction | optimises variance, not target accuracy |
| K-means | discover groups | needs $K$, local minima |
| Hierarchical clustering | nested grouping | linkage/distance sensitivity |

# 19. Common mistakes

1. Reporting training performance as generalisation performance.
2. Scaling or imputing before splitting.
3. Tuning hyperparameters on the final test set.
4. Assuming greater flexibility is automatically better.
5. Interpreting predictive association causally.
6. Forgetting scaling for KNN, PCA and K-means.
7. Treating PCA as ordinary feature selection.
8. Treating cluster labels as discovered ground truth.

# 20. Student exercises

### Exercise A — species-aware regression

Add species to the body-mass model. Compare RMSE and slope interpretation.

### Exercise B — interaction effects

Fit a flipper-length × species interaction and ask whether slopes differ across species.

### Exercise C — nested cross-validation

Use inner CV for tuning and outer CV for unbiased evaluation of the tuning procedure.

### Exercise D — leakage experiment

Compare PCA fitted before CV against PCA placed inside a pipeline.

### Exercise E — clustering sensitivity

Compare K-means on raw features, standardized features and PCA scores.

# 21. Closing perspective

ST3248 is fundamentally about balancing

$$
\\text{fit} \\quad \\text{vs} \\quad \\text{generalisation},
$$

$$
\\text{bias} \\quad \\text{vs} \\quad \\text{variance},
$$

$$
\\text{flexibility} \\quad \\text{vs} \\quad \\text{stability},
$$

and

$$
\\text{prediction} \\quad \\text{vs} \\quad \\text{interpretability}.
$$

For every method, ask:

1. What quantity is being optimised?
2. What assumptions are being made?
3. How is generalisation estimated?
4. How can the method fail?

# References

- NUS ST3248 Statistical Learning I course/module materials
- James, Witten, Hastie, Tibshirani & Taylor, *An Introduction to Statistical Learning*
- Palmer Penguins dataset
- scikit-learn documentation
- Bokeh documentation

Dataset URL used in this notebook:

`https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv`